# Simple Demonstration of using ToolCalls with OpenAI
The example: get Weather via ToolCall

In [29]:
from dotenv import load_dotenv
from openai import OpenAI
import json

In [30]:
load_dotenv()

True

In [31]:
openai = OpenAI()

In [32]:
def get_weather(city):
    print("GOT CITY: ", city)
    if city.lower() == "los angeles":
        return {"unit":"farenheit", "value":"85"}
    elif city.lower() == "san diego":
        return {"unit":"farenheit", "value":"78"}
    elif city.lower() == "miami":
        return {"unit":"farenheit", "value":"90"}

    return {"unit":"celcius", "value":"10000"}

In [33]:
def is_valid_city(city):
    # a list of valid cities
    valid_cities = [
        "Salt Lake City",
        "Jacksonville",
        "Boise",
        "Charleston",
        "San Jose",
        "Buttzville",
        "Cleveland",
        "Kansas City",
        "Roswell",
        "Frankenmuth",
        "Monowi",
        "Casey",
        "Albany",
        "Salt Lake City",
        "Baltimore",
        "Cincinnati",
    ]

    return {"is_valid":city in valid_cities}

In [34]:
get_weather_json = {
    "name": "get_weather",
    "description": "Use this tool to obtain the weather for any given city",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "Name of city to get weather for",
            },
        },
        "required": ["city"],
        "additionalProperties": False,
    }
}

is_valid_city_json = {
    "name": "is_valid_city",
    "description": "Use this tool to determine if the city is valid",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "Name of city to verify for validity",
            },
        },
        "required": ["city"],
        "additionalProperties": False,
    }
}

In [35]:
tools = [
    {"type":"function", "function":get_weather_json},{"type":"function", "function":is_valid_city_json}]

In [36]:
def handle_tool_calls(tools):
    # loop through all tools and call the appropriate tool
    tool_results = []
    for tool in tools:
        # get required metadata
        fn = globals().get(tool.function.name)
        args = json.loads(tool.function.arguments)
        print("CALLING FUNCTION: ", fn, " ARGS: ", args)
        print(fn, args)
         # call the tool
        result = fn(**args) # why the double pointer, though? 

         # append to list
        tool_results.append({"role":"tool", "content":json.dumps(result), "tool_call_id": tool.id})

    # return the result
    return tool_results

In [37]:
system_prompt = "You are a meterologist.  You will determine the weather for the provide city after verifying it is a valid city.  If it is not a valid city, tell the user it is not a valid city and do not try to determine its weather; the tools you will use are is_valid_city and get_weather"
#system_prompt = "You are a meterologist.  You will determine the weather for the provided city; the tool to use is get_weather"

In [38]:
# chat callback for gradio
def chatCallback(message, history):
    # consolidate system message, history, and current message
    messages = [{"role": "system", "content":system_prompt}] + history + [{"role":"user", "content":message}]
    responseComplete = False

    while not responseComplete:
        response = openai.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            tools=tools,
        )

        print("got response")
        finish_reason = response.choices[0].finish_reason
        response_message = response.choices[0].message
        if finish_reason == "tool_calls":
            # use list of tools to handle tool calls
            tool_call_results = handle_tool_calls(response_message.tool_calls)

            # update messages to include latest content from the model, plus tool_call results
            messages.append(response_message)
            messages.extend(tool_call_results)

            # must loop to call api again
            print("looping again")
            print(messages)
        else:
            # no need for tool calls;
            responseComplete = True

    return response.choices[0].message.content
    

In [39]:
import gradio as gr
gr.ChatInterface(chatCallback, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


got response
got response
CALLING FUNCTION:  <function is_valid_city at 0x7bbd0f6836a0>  ARGS:  {'city': 'los angeles'}
<function is_valid_city at 0x7bbd0f6836a0> {'city': 'los angeles'}
looping again
[{'role': 'system', 'content': 'You are a meterologist.  You will determine the weather for the provide city after verifying it is a valid city.  If it is not a valid city, tell the user it is not a valid city and do not try to determine its weather; the tools you will use are is_valid_city and get_weather'}, {'role': 'user', 'metadata': None, 'content': 'who are you', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': "I am a meteorologist here to help you determine the weather for any valid city you provide. If you'd like to know the weather in a specific city, just let me know!", 'options': None}, {'role': 'user', 'content': 'los angeles'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[C